In [1]:
# Gerekli kütüphaneleri içe aktarıyoruz
from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    col, current_timestamp, when,
    count, avg, round
)
from pyspark.sql.types import (
    StructType, StructField,
    StringType, IntegerType,
    FloatType, BooleanType
)
from delta import configure_spark_with_delta_pip
import os

print("✅ Kütüphaneler başarıyla yüklendi!")

✅ Kütüphaneler başarıyla yüklendi!


In [2]:
# Spark oturumunu Delta Lake desteğiyle başlatıyoruz
builder = (
    SparkSession.builder
    .appName("SpotifyBigDataProject")
    .config("spark.sql.extensions",
            "io.delta.sql.DeltaSparkSessionExtension")
    .config("spark.sql.catalog.spark_catalog",
            "org.apache.spark.sql.delta.catalog.DeltaCatalog")
    .config("spark.sql.shuffle.partitions", "4")
)

spark = configure_spark_with_delta_pip(builder).getOrCreate()
spark.sparkContext.setLogLevel("ERROR")

print("✅ Spark oturumu başarıyla başlatıldı!")
print("Spark versiyonu:", spark.version)

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/05/09 20:03:57 WARN Utils: Your hostname, Ceylo-MacBook-Pro.local, resolves to a loopback address: 127.0.0.1; using 192.168.1.183 instead (on interface en0)
26/05/09 20:03:57 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
:: loading settings :: url = jar:file:/Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/site-packages/pyspark/jars/ivy-2.5.3.jar!/org/apache/ivy/core/settings/ivysettings.xml
Ivy Default Cache set to: /Users/ceylo/.ivy2.5.2/cache
The jars for the packages stored in: /Users/ceylo/.ivy2.5.2/jars
io.delta#delta-spark_4.1_2.13 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-a9d31f27-fd50-452c-aeb8-51bab6f516d3;1.0
	confs: [default]
	found io.delta#delta-spark_4.1_2.13;4.2.0 in central
	found io.delta#delta-storage;4.2.0 in central
	found io.unitycatalog#unitycatalog-client;0.4.1 in central
	found org.slf4j#slf4j

✅ Spark oturumu başarıyla başlatıldı!
Spark versiyonu: 4.1.1


In [3]:
# Proje klasör yollarını tanımlıyoruz
import os

PROJE_KLASORU = os.path.expanduser("~/spotify-bigdata-project")

BRONZE_YOL = f"{PROJE_KLASORU}/delta-lake/bronze"
SILVER_YOL = f"{PROJE_KLASORU}/delta-lake/silver"
GOLD_YOL   = f"{PROJE_KLASORU}/delta-lake/gold"
HAM_VERI_YOL = f"{PROJE_KLASORU}/data/raw"

print("✅ Klasör yolları tanımlandı!")
print(f"Bronze : {BRONZE_YOL}")
print(f"Silver : {SILVER_YOL}")
print(f"Gold   : {GOLD_YOL}")
print(f"Ham Veri: {HAM_VERI_YOL}")

✅ Klasör yolları tanımlandı!
Bronze : /Users/ceylo/spotify-bigdata-project/delta-lake/bronze
Silver : /Users/ceylo/spotify-bigdata-project/delta-lake/silver
Gold   : /Users/ceylo/spotify-bigdata-project/delta-lake/gold
Ham Veri: /Users/ceylo/spotify-bigdata-project/data/raw


In [4]:
# Spotify veri setinin şemasını tanımlıyoruz
# dataset.csv - 114K şarkı, 21 kolon
spotify_sema = StructType([
    StructField("_c0",               IntegerType(), True),  # index kolonu
    StructField("track_id",          StringType(),  True),
    StructField("artists",           StringType(),  True),
    StructField("album_name",        StringType(),  True),
    StructField("track_name",        StringType(),  True),
    StructField("popularity",        IntegerType(), True),
    StructField("duration_ms",       IntegerType(), True),
    StructField("explicit",          StringType(),  True),
    StructField("danceability",      FloatType(),   True),
    StructField("energy",            FloatType(),   True),
    StructField("key",               IntegerType(), True),
    StructField("loudness",          FloatType(),   True),
    StructField("mode",              IntegerType(), True),
    StructField("speechiness",       FloatType(),   True),
    StructField("acousticness",      FloatType(),   True),
    StructField("instrumentalness",  FloatType(),   True),
    StructField("liveness",          FloatType(),   True),
    StructField("valence",           FloatType(),   True),
    StructField("tempo",             FloatType(),   True),
    StructField("time_signature",    IntegerType(), True),
    StructField("track_genre",       StringType(),  True),
])

print("✅ Şema tanımlandı!")
print(f"Toplam kolon sayısı: {len(spotify_sema.fields)}")

✅ Şema tanımlandı!
Toplam kolon sayısı: 21


In [5]:
# Ham veriyi CSV'den okuyoruz
# Kafka simülasyonu için şimdilik CSV kullanıyoruz
# Kişi 1 Kafka'yı bitirince bu kısım güncellenecek
ham_veri = (
    spark.read
    .option("header", "true")
    .option("inferSchema", "false")
    .schema(spotify_sema)
    .csv(f"{HAM_VERI_YOL}/dataset.csv")
)

# _c0 index kolonunu düşürüyoruz, işimize yaramıyor
ham_veri = ham_veri.drop("_c0")

print(f"✅ Ham veri okundu!")
print(f"Toplam satır sayısı: {ham_veri.count():,}")
print(f"Kolon sayısı: {len(ham_veri.columns)}")
ham_veri.show(3)

✅ Ham veri okundu!
Toplam satır sayısı: 114,000
Kolon sayısı: 20
+--------------------+--------------------+----------------+----------------+----------+-----------+--------+------------+------+---+--------+----+-----------+------------+----------------+--------+-------+------+--------------+-----------+
|            track_id|             artists|      album_name|      track_name|popularity|duration_ms|explicit|danceability|energy|key|loudness|mode|speechiness|acousticness|instrumentalness|liveness|valence| tempo|time_signature|track_genre|
+--------------------+--------------------+----------------+----------------+----------+-----------+--------+------------+------+---+--------+----+-----------+------------+----------------+--------+-------+------+--------------+-----------+
|5SuOikwiRyPMVoIQD...|         Gen Hoshino|          Comedy|          Comedy|        73|     230666|   False|       0.676| 0.461|  1|  -6.746|   0|      0.143|      0.0322|         1.01E-6|   0.358|  0.715|87.917

In [6]:
# Ham veriyi Kafka simülasyonu için timestamp ekleyip Bronze'a yazıyoruz
# kafka_timestamp: Kafka'dan geliyormuş gibi simüle ediyoruz
bronze_veri = ham_veri.withColumn(
    "kafka_timestamp", current_timestamp()
)

# Delta formatında Bronze katmanına yazıyoruz
(
    bronze_veri.write
    .format("delta")
    .mode("overwrite")
    .save(BRONZE_YOL)
)

print("✅ Bronze katmanına yazıldı!")
print(f"Konum: {BRONZE_YOL}")

# Doğrulama: geri okuyup kontrol ediyoruz
bronze_kontrol = spark.read.format("delta").load(BRONZE_YOL)
print(f"Bronze satır sayısı: {bronze_kontrol.count():,}")

✅ Bronze katmanına yazıldı!
Konum: /Users/ceylo/spotify-bigdata-project/delta-lake/bronze
Bronze satır sayısı: 114,000


In [7]:
# Silver katmanı için veri temizleme yapıyoruz

# Temizlemeden önce null sayılarını kontrol ediyoruz
print("=== TEMİZLEME ÖNCESİ NULL SAYILARI ===")
bronze_veri.select([
    count(when(col(c).isNull(), c)).alias(c)
    for c in bronze_veri.columns
]).show()

# Adım 1: Kritik kolonlarda null varsa o satırı at
temiz_veri = bronze_veri.dropna(
    subset=["track_id", "track_name", "track_genre", "popularity"]
)
print(f"Null temizleme sonrası: {temiz_veri.count():,}")

# Adım 2: Duplike kayıtları kaldır
temiz_veri = temiz_veri.dropDuplicates(["track_id"])
print(f"Duplike temizleme sonrası: {temiz_veri.count():,}")

# Adım 3: Popularity değeri 0-100 dışındaysa filtrele
temiz_veri = temiz_veri.filter(
    (col("popularity") >= 0) & (col("popularity") <= 100)
)
print(f"Popularity filtresi sonrası: {temiz_veri.count():,}")

# Adım 4: explicit kolonunu True/False'dan 0/1'e çevir
temiz_veri = temiz_veri.withColumn(
    "explicit",
    when(col("explicit") == "True", 1)
    .when(col("explicit") == "False", 0)
    .otherwise(0)
)

print("\n✅ Veri temizleme tamamlandı!")

=== TEMİZLEME ÖNCESİ NULL SAYILARI ===
+--------+-------+----------+----------+----------+-----------+--------+------------+------+---+--------+----+-----------+------------+----------------+--------+-------+-----+--------------+-----------+---------------+
|track_id|artists|album_name|track_name|popularity|duration_ms|explicit|danceability|energy|key|loudness|mode|speechiness|acousticness|instrumentalness|liveness|valence|tempo|time_signature|track_genre|kafka_timestamp|
+--------+-------+----------+----------+----------+-----------+--------+------------+------+---+--------+----+-----------+------------+----------------+--------+-------+-----+--------------+-----------+---------------+
|       0|      1|         1|         1|       134|         36|       0|         107|    33|130|       5| 107|          4|           3|               0|       1|      2|    0|           132|          0|              0|
+--------+-------+----------+----------+----------+-----------+--------+------------+

In [8]:
# Temizlenmiş veriyi Silver katmanına yazıyoruz
(
    temiz_veri.write
    .format("delta")
    .mode("overwrite")
    .save(SILVER_YOL)
)

print("✅ Silver katmanına yazıldı!")
print(f"Konum: {SILVER_YOL}")

# Doğrulama
silver_kontrol = spark.read.format("delta").load(SILVER_YOL)
print(f"Silver satır sayısı: {silver_kontrol.count():,}")
silver_kontrol.show(3)

✅ Silver katmanına yazıldı!
Konum: /Users/ceylo/spotify-bigdata-project/delta-lake/silver
Silver satır sayısı: 89,620
+--------------------+--------------------+----------+--------------------+----------+-----------+--------+------------+------+---+--------+----+-----------+------------+----------------+--------+-------+-------+--------------+--------------+--------------------+
|            track_id|             artists|album_name|          track_name|popularity|duration_ms|explicit|danceability|energy|key|loudness|mode|speechiness|acousticness|instrumentalness|liveness|valence|  tempo|time_signature|   track_genre|     kafka_timestamp|
+--------------------+--------------------+----------+--------------------+----------+-----------+--------+------------+------+---+--------+----+-----------+------------+----------------+--------+-------+-------+--------------+--------------+--------------------+
|000Iz0K615UepwSJ5...|Paul Kalkbrenner;...|         X|Böxig Leise - Pig...|        22|    

In [9]:
# Gold katmanı: Tür (track_genre) bazında özet istatistikler
# Analiz için hazır veri oluşturuyoruz
gold_veri = (
    silver_kontrol
    .groupBy("track_genre")
    .agg(
        count("track_id").alias("sarki_sayisi"),
        round(avg("popularity"), 2).alias("ort_popularity"),
        round(avg("danceability"), 4).alias("ort_danceability"),
        round(avg("energy"), 4).alias("ort_energy"),
        round(avg("tempo"), 2).alias("ort_tempo"),
        round(avg("loudness"), 2).alias("ort_loudness"),
        round(avg("valence"), 4).alias("ort_valence")
    )
    .orderBy(col("ort_popularity").desc())
)

# Gold katmanına yazıyoruz
(
    gold_veri.write
    .format("delta")
    .mode("overwrite")
    .save(GOLD_YOL)
)

print("✅ Gold katmanına yazıldı!")

# Doğrulama
gold_kontrol = spark.read.format("delta").load(GOLD_YOL)
print(f"Toplam tür sayısı: {gold_kontrol.count()}")
print("\nEn popüler 10 tür:")
gold_kontrol.show(10)

✅ Gold katmanına yazıldı!
Toplam tür sayısı: 113

En popüler 10 tür:
+-----------+------------+--------------+----------------+----------+---------+------------+-----------+
|track_genre|sarki_sayisi|ort_popularity|ort_danceability|ort_energy|ort_tempo|ort_loudness|ort_valence|
+-----------+------------+--------------+----------------+----------+---------+------------+-----------+
|      k-pop|         916|         59.42|          0.6418|    0.6828|   119.53|       -6.42|      0.569|
|   pop-film|         813|          59.1|          0.5914|    0.5997|   116.97|       -7.87|     0.5288|
|      metal|         232|         56.42|          0.4812|    0.8414|   129.48|       -4.93|     0.4249|
|      chill|         972|         53.74|          0.6664|    0.4295|   115.38|      -10.41|     0.4083|
|     latino|         398|         51.79|          0.7555|    0.7123|   121.42|       -5.55|     0.6225|
|        sad|         564|         51.11|          0.7018|    0.4789|   119.36|       -9.72

In [10]:
# Tüm katmanların özetini yazdırıyoruz
print("=" * 50)
print("PROJE ÖZET RAPORU")
print("=" * 50)

bronze_df = spark.read.format("delta").load(BRONZE_YOL)
silver_df = spark.read.format("delta").load(SILVER_YOL)
gold_df   = spark.read.format("delta").load(GOLD_YOL)

print(f"🥉 Bronze katmanı : {bronze_df.count():>8,} satır (ham veri)")
print(f"🥈 Silver katmanı : {silver_df.count():>8,} satır (temizlenmiş)")
print(f"🥇 Gold katmanı   : {gold_df.count():>8,} satır (tür bazında özet)")
print(f"\nTemizlenen satır : {bronze_df.count() - silver_df.count():,}")
print(f"Toplam tür sayısı: {gold_df.count()}")
print("=" * 50)
print("✅ Adım 3 tamamlandı! Delta Lake hazır.")

PROJE ÖZET RAPORU
🥉 Bronze katmanı :  114,000 satır (ham veri)
🥈 Silver katmanı :   89,620 satır (temizlenmiş)
🥇 Gold katmanı   :      113 satır (tür bazında özet)

Temizlenen satır : 24,380
Toplam tür sayısı: 113
✅ Adım 3 tamamlandı! Delta Lake hazır.


In [11]:
print(silver_kontrol.columns)

['track_id', 'artists', 'album_name', 'track_name', 'popularity', 'duration_ms', 'explicit', 'danceability', 'energy', 'key', 'loudness', 'mode', 'speechiness', 'acousticness', 'instrumentalness', 'liveness', 'valence', 'tempo', 'time_signature', 'track_genre', 'kafka_timestamp']
